In [2]:

import numpy as np
import time
import myQMLlib as myQML
import json
import os
import matplotlib.pyplot as plt

N_train_list = [1,2,3,4,5,6,7,8,9,10,15,20,25,30,35,40,45,50]  
N_test = 250
num_realizations = 100
num_shots = 0
filename = "qelm_test.json"

# --- LOAD PREVIOUS DATA IF IT EXISTS ---
if os.path.exists(filename):
    print(f"Found existing file '{filename}'. Loading previous data...")
    with open(filename, 'r') as f:
        saved_data = json.load(f)
else:
    print("No previous data found. Starting fresh.")
    saved_data = {'QELM_1': {}, 'QELM_2': {}}

# Filter N_train_list to ONLY calculate values we haven't done yet
# (JSON saves dictionary keys as strings, so we check str(N))
N_train_to_run = [N for N in N_train_list if str(N) not in saved_data['QELM_1']]

if not N_train_to_run:
    print("All N_train values in the list have already been simulated. Exiting.")
    exit()

print(f"Simulating for N_train = {N_train_to_run}")

N_max = max(N_train_to_run)

# QELM reservoir physical setup
d_in = 2
d_res = 64
d_out = d_in * d_res
povm = myQML.generate_computational_povm(d_res) 
num_povm_elements = len(povm)

# Temporary dictionary for the CURRENT run
mse_runs = {
    'QELM_1': {N: [] for N in N_train_to_run},
    'QELM_2': {N: [] for N in N_train_to_run},
}

print(f"-------- (Shots = {num_shots}, Realizations = {num_realizations}) --------")
print("="*70)

total_time = time.time()


# INVERTED LOOP
for r in range(num_realizations):
    if (r + 1) % 10 == 0:
        print(f"Processing Realization {r + 1}/{num_realizations}...")
        
    ds = myQML.QuantumDatasetGenerator(N_max, N_test, myQML.X)
    ds.generate_density_matrices_vec()
    ds.compute_expectation_values_vec()

    rho_train_full, y_train_full = ds.get_training_dataset()
    rho_test, y_test = ds.get_test_dataset()

    #putting V here, we are assuming that for each realization the input state of
    #the reservoir is the same, and that the only thing that changes is the training/test states
    V = myQML.random_isometry(d_in, d_in * d_res)
    qelm = myQML.QuantumExtremeLearningMachine(
        isometry=V, povm=povm, bipartite_dims=(d_in, d_res), 
        keep_subsystem=1, num_shots=num_shots
    )

    qelm2 = myQML.QuantumExtremeLearningMachine(
        isometry=V, povm=povm, bipartite_dims=(d_in, d_res), 
        keep_subsystem=1, num_shots=num_shots
    )

    for N_train in sorted(N_train_to_run, reverse=True):
        rho_train_sub = rho_train_full[:N_train]
        y_train_sub = y_train_full[:N_train]

        # QELM
        qelm.fit_vec(rho_train_sub, y_train_sub)
        qelm2.fit_vec_2(rho_train_sub, y_train_sub)
        mse_runs['QELM_1'][N_train].append(float(np.mean((qelm.predict_vec(rho_test) - y_test)**2)))
        mse_runs['QELM_2'][N_train].append(float(np.mean((qelm2.predict_vec(rho_test) - y_test)**2)))

        if num_realizations == 1 and N_train == N_train_to_run[-1]:
            print("\n--- Testing Features ---")
            t0 = time.time()
            P1 = qelm.get_features_vec(rho_train_sub)
            t1 = time.time()
    
            t2 = time.time()
            P2 = qelm2.get_features_vec_2(rho_train_sub)
            t3 = time.time()

            print(f"P1 (vec)   shape: {P1.shape} | Time: {t1-t0:.4f}s")
            print(f"P2 (vec_2) shape: {P2.shape} | Time: {t3-t2:.4f}s")
    
             # P1 should be (N, M). P2 should be (M, N). Therefore P1.T should equal P2.
            np.testing.assert_allclose(P1.T, P2, atol=1e-10, 
                               err_msg="Features do not match!")
            print("✅ Features match perfectly (P1.T == P2).")

            print("\n--- Testing Fit (Weights) ---")
            qelm.fit_vec(rho_train_sub, y_train_sub)
            W1 = qelm.W.copy()
    
            qelm2.fit_vec_2(rho_train_sub, y_train_sub)
            W2 = qelm2.W.copy()
    
            print(f"W1 (vec)   shape: {W1.shape}")
            print(f"W2 (vec_2) shape: {W2.shape}")
    
             # Because both methods solve for the same physical weights mapping POVMs to labels, 
            # the resulting weights should be exactly identical!
            np.testing.assert_allclose(W1, W2, atol=1e-10, 
                               err_msg="Trained weights do not match!")
            print("✅ Weights match perfectly (W1 == W2).")

            # 4. Test Predictions
            print("\n--- Testing Predictions ---")
            # Restore W1 to test predict_vec (since fit_vec_2 overwrote self.W)
            qelm.W = W1 
            y_pred_1 = qelm.predict_vec(rho_test)
    
            qelm2.W = W2
            y_pred_2 = qelm2.predict_vec_2(rho_test)
    
            print(f"Pred 1 shape: {y_pred_1.shape}")
            print(f"Pred 2 shape: {y_pred_2.shape}")
    
            np.testing.assert_allclose(y_pred_1, y_pred_2, atol=1e-10, 
                               err_msg="Predictions do not match!")
            print("✅ Predictions match perfectly (y_pred_1 == y_pred_2).")
    
            print("\n=== All Benchmarks Passed Successfully! ===")




# --- MERGE AND SAVE DATA ---
for model in mse_runs.keys():
    for N_train, mse_list in mse_runs[model].items():
        # JSON requires dictionary keys to be strings
        saved_data[model][str(N_train)] = mse_list

with open(filename, 'w') as f:
    json.dump(saved_data, f, indent=4)

print("="*70)
print(f"Experiment finished and data appended to '{filename}' in {(time.time() - total_time)/60:.1f} minutes.")

No previous data found. Starting fresh.
Simulating for N_train = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 35, 40, 45, 50]
-------- (Shots = 0, Realizations = 100) --------
Processing Realization 10/100...
Processing Realization 20/100...
Processing Realization 30/100...
Processing Realization 40/100...
Processing Realization 50/100...
Processing Realization 60/100...
Processing Realization 70/100...
Processing Realization 80/100...
Processing Realization 90/100...
Processing Realization 100/100...
Experiment finished and data appended to 'qelm_test.json' in 31.4 minutes.
